# 01 - Data Screening

**Notebook version:** v5 -- 2026-07-29

Load raw SECOM data, inspect missingness/variance, and build the data dictionary.

The first call to `load_raw()` below will automatically download the raw SECOM
data from the UCI Machine Learning Repository into `data/raw/` if it isn't
already present -- no separate script needs to be run first. This requires
internet access and may take a few seconds the first time.

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# Also pulls latest changes and prints the commit hash, so you can confirm at a
# glance (against GitHub's commit history) that you're looking at the current version.
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
    else:
        os.chdir(f"/content/{REPO_NAME}")
        !git pull
        os.chdir("/content")
    os.chdir(f"/content/{REPO_NAME}/notebooks")
    !pip install -q -r ../requirements.txt

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import pandas as pd
from preprocessing import load_raw, screen_missingness, screen_variance

X, y = load_raw()
print(X.shape, y.shape)
y.value_counts()

## Missingness and variance profile

Fill these numbers into `docs/data_dictionary_template.csv`.

In [ ]:
missing_frac = X.isna().mean().sort_values(ascending=False)
missing_frac.head(20)

In [ ]:
variances = X.var(numeric_only=True).sort_values()
variances.head(20)

In [ ]:
corr_with_target = X.corrwith(y.astype(float)).abs().sort_values(ascending=False)
corr_with_target.head(20)

## Apply screening thresholds

Default thresholds: drop columns with >40% missing values or near-zero variance.
Adjust and document the final thresholds used in the capstone report.

In [ ]:
X_screened = screen_missingness(X, max_missing_frac=0.4)
X_screened = screen_variance(X_screened, min_variance=1e-6)
print(f"Remaining features: {X_screened.shape[1]} (from {X.shape[1]})")

## Next steps

- Export screening results into `docs/data_dictionary_template.csv`
- Proceed to `02_modeling_rq1.ipynb` for baseline + imbalance-corrected models

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "01_data_screening"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed.
    from google.colab import _message
    ipynb_content = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
    with open(export_path, 'w') as f:
        json.dump(ipynb_content, f)

html_output = f"{NOTEBOOK_NAME}.html"
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print(f"Exported to {html_output}")

if IN_COLAB and result.returncode == 0:
    from google.colab import files
    files.download(html_output)
